<a href="https://colab.research.google.com/github/Alt4660/con5200-card-replication/blob/main/notebooks/01_positioning_and_replication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sections 1 and 2: Positioning and Direct Replication

ECON 5200 Midterm Project · Replication Study

> **🛑 AI policy, and it is not the problem sets' policy.** Sections 1 and 2 — the
> positioning and the direct replication — are **no AI**. Not for the code, not for
> the prose. Making a computer agree with a published table is the exercise that
> establishes you can read a paper, and a model that has already read the paper will
> simply hand you the answer without your ever finding the discrepancy. Sections 3 to
> 6 authorise AI as a documented co-pilot under **P.R.I.M.E.**, on the same terms as
> every problem set: you file the full transcript in `ai-appendix.md`, and if you
> cannot explain a line you cannot submit it. State in your README which sections you
> used it in. This follows the course rule — manual command first, expansion second —
> and using AI in Sections 1 or 2 is an integrity violation, not a style disagreement.

## Setup

Imports and paths only.

In [74]:
import sys, os, subprocess
if "google.colab" in sys.modules and not os.path.exists("download_data.py"):
    subprocess.run(["git", "clone",
                    "https://github.com/Alt4660/con5200-card-replication"], check=True)
    os.chdir("con5200-card-replication")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "linearmodels"], check=True)

In [75]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from linearmodels.iv import IV2SLS

# The repository root, whether this runs from notebooks/ (Jupyter) or from the root (reproduce.py).
ROOT = Path.cwd() if (Path.cwd() / "download_data.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
(ROOT / "output").mkdir(exist_ok=True)

from download_data import MY_PAPER, PAPERS, fetch

## Data

In [76]:
# Set MY_PAPER once, in download_data.py. fetch() downloads into data/ only if a file is missing.
if MY_PAPER:
    paths = fetch(MY_PAPER)
    print(PAPERS[MY_PAPER]["paper"])
    df = pd.read_stata(paths[0])
    print(df.shape)
else:
    print("TODO: set MY_PAPER in download_data.py (castle, card, lee, dw, hansen or abortion).")

  data/card.dta  0.5 MB
Card (1995), proximity to college and returns to schooling
(3010, 37)


## Section 1: Positioning (12 pts)

Two pages maximum in the report. What question the paper asks, what was believed before it, what it changed, and how it has been received, with at least three citations to work that engages with it critically. Then the identification strategy and the assumption it rests on, in your own words.

The prose goes in the report. Keep your notes and the three citations here.

**Critical citations (at least three):**

1. TODO
2. TODO
3. TODO

**The identification strategy and its assumption, in my own words:**

*TODO*

## Section 2: Direct Replication (24 pts)

Reproduce the paper's headline result and report your number beside theirs. You will not match exactly. Document every discrepancy and its likely source: sample construction, variable definition, software defaults, a decision the paper does not describe. Where the paper is ambiguous about a choice, say so: those ambiguities are your Section 3.

In [77]:
df["agesq"] = df["age"] ** 2
for col in ["fatheduc", "motheduc"]:
    df[f"{col}_miss"] = df[col].isna().astype(int)
    df[col] = df[col].fillna(df[col].mean())
FAMILY_BASE = ("fatheduc + motheduc + fatheduc_miss + motheduc_miss"
               " + momdad14 + sinmom14")

print(df[["fatheduc_miss", "motheduc_miss"]].sum())

fatheduc_miss    690
motheduc_miss    353
dtype: int64


In [78]:


CUTS = [-1, 11, 12, 30]
LABELS = ["lt12", "hs", "morehs"]

df["mom_cls"] = pd.cut(df["motheduc"], bins=CUTS, labels=LABELS)
df["dad_cls"] = pd.cut(df["fatheduc"], bins=CUTS, labels=LABELS)


combo = df["mom_cls"].astype(str) + "_" + df["dad_cls"].astype(str)
dummies = pd.get_dummies(combo, prefix="pe", drop_first=True).astype(int)
df = df.drop(columns=[c for c in df.columns if c.startswith("pe_")])
df = pd.concat([df, dummies], axis=1)

FAMILY = FAMILY_BASE + " + " + " + ".join(dummies.columns)

REGIONS = " + ".join(f"reg66{i}" for i in range(2, 10))

CONTROLS = f"black + south + smsa + smsa66 + {REGIONS} + {FAMILY}"

print(len(dummies.columns))
print(CONTROLS)


8
black + south + smsa + smsa66 + reg662 + reg663 + reg664 + reg665 + reg666 + reg667 + reg668 + reg669 + fatheduc + motheduc + fatheduc_miss + motheduc_miss + momdad14 + sinmom14 + pe_hs_lt12 + pe_hs_morehs + pe_lt12_hs + pe_lt12_lt12 + pe_lt12_morehs + pe_morehs_hs + pe_morehs_lt12 + pe_morehs_morehs


In [79]:
ols = smf.ols(f"lwage ~ educ + exper + expersq + {CONTROLS}", data=df).fit()

print("OLS education coef:", round(ols.params["educ"], 4))
print("OLS standard error:", round(ols.bse["educ"], 4))
print("N:", int(ols.nobs))

OLS education coef: 0.0721
OLS standard error: 0.0037
N: 3010


In [80]:

COV = "unadjusted"

iv = IV2SLS.from_formula(
    f"lwage ~ 1 + {CONTROLS} + [educ + exper + expersq ~ nearc4 + age + agesq]",
    data=df,
).fit(cov_type=COV)


iv_simple = IV2SLS.from_formula(
    f"lwage ~ 1 + exper + expersq + {CONTROLS} + [educ ~ nearc4]",
    data=df,
).fit(cov_type=COV)

print("IV education coef:", round(iv.params["educ"], 4))
print("IV standard error:", round(iv.std_errors["educ"], 4))
print("IV simple version:", round(iv_simple.params["educ"], 4))

IV education coef: 0.1314
IV standard error: 0.0469
IV simple version: 0.14


In [81]:
fs = smf.ols(f"educ ~ nearc4 + age + agesq + {CONTROLS}", data=df).fit()
f_headline = iv.first_stage.diagnostics.loc["educ", "f.stat"]
f_simple = iv_simple.first_stage.diagnostics.loc["educ", "f.stat"]

print("First stage, nearc4 coef:", round(fs.params["nearc4"], 4))
print("F (headline, 3 instruments):", round(f_headline, 2))
print("F (simple, nearc4 only):", round(f_simple, 2))

First stage, nearc4 coef: 0.3803
F (headline, 3 instruments): 8.41
F (simple, nearc4 only): 16.19


### Theirs vs mine

One row per number you replicate. The row label says where the number is in the paper (for example, `Table 2, col 3: effect on log wage`). `gap_pct` is computed for you as 100 × (mine − theirs) / |theirs|. This cell writes `output/headline_table.csv`, which `python reproduce.py` prints and your README quotes.

In [82]:
headline = pd.DataFrame(
    {
        "paper_estimate": pd.Series(dtype="float"),
        "my_estimate": pd.Series(dtype="float"),
        "gap_pct": pd.Series(dtype="float"),
        "likely_source": pd.Series(dtype="object"),
    },
    index=pd.Index([], name="estimate", dtype="object"),
)


headline.loc["Table 4 row 1: OLS return to education"] = [
    0.073, ols.params["educ"], np.nan,
    " Theres a small gap but its the same data, but my parent-education controls may not match the paper's"]

headline.loc["Table 3 Panel B col 6: IV return to education"] = [
    0.132, iv.params["educ"], np.nan,
    "Parent-education class cutoffs"]

headline.loc["Table 3 Panel B col 2: first stage, nearc4 on education"] = [
    0.365, fs.params["nearc4"], np.nan,
    "Likely the same issue my interaction dummies pull this up to about 0.38"]

headline["gap_pct"] = 100 * (headline["my_estimate"] - headline["paper_estimate"]) / headline["paper_estimate"].abs()
headline.to_csv(ROOT / "output" / "headline_table.csv")
headline

,paper_estimate,my_estimate,gap_pct,likely_source
estimate,,,,
Table 4 row 1: OLS return to education,0.073,0.072086,-1.252452,"Theres a small gap but its the same data, but..."
Table 3 Panel B col 6: IV return to education,0.132,0.131399,-0.455192,Parent-education class cutoffs
"Table 3 Panel B col 2: first stage, nearc4 on education",0.365,0.380286,4.188040,Likely the same issue my interaction dummies p...


| # | Estimate (row above) | What differs | Size of the gap | Likely source | How I checked it | Is the paper ambiguous here? |
|---|---|---|---|---|---|---|
| 1 | IV, return to education | Mine is 0.1314, paper is 0.132 | -0.0006 (-0.5%). My first version was off by -3% | I suspect it's how the parent-education controls are built, because it's the main part of my code where I had to guess | I started with no interaction controls (IV 0.1281). I then dropped all the family controls to see their effect, and it moved IV further away (0.1224). I then added the interactions of mother's and father's education, which got IV to 0.1306 with 4 classes per parent and 0.1314 with 3, so I got even closer with 3 classes | Yes. The paper says "8 classes" but never gives the cutoffs |
| 2 | IV, other cutoffs | I changed the cutoffs a second time (under 12 / 12 to 15 / 16 or more) and IV came out at 0.1335, so this time it overshot the paper | +0.0015 (+1.1%) | Same thing: the way I sort parents into education classes | I wanted to see whether the result just moves around depending on where I put the cutoffs, or whether it settles. It moves around: across my three versions IV goes 0.1306, 0.1314, 0.1335, and the paper's 0.132 sits right in the middle of that range. So I can't say one grouping is "the" right one, only that this choice is enough to explain a gap this size | Yes. The paper gives no cutoffs, so any of these could be what the authors did |
| 3 | First stage, nearc4 | Mine is 0.3803 and the paper's is 0.365, so the nudge looks a bit stronger in my version | +0.0153 (+4.2%) | Probably the same cause, but I'm less sure here. My guess is that my interaction dummies soak up a bit more of the nudge than the paper's do, which pushes the number up | Before I added the interactions my first stage was 0.3653, almost exactly the paper's, which confused me because IV was still off. Once I added the 8 interaction dummies it moved to between 0.360 and 0.380 depending on the cutoffs. So the first stage and IV don't move together, and I haven't found a grouping that matches both. That makes me think the authors built their dummies differently from how I did | Yes. I don't know how the paper builds its dummies, only that it uses "8 classes" and 14 family variables in total |
| 4 | OLS, return to education | Mine is 0.0721 and the paper's is 0.073, a small gap | -0.0009 (-1.25%) | Probably the same controls issue, but it's tiny | Across every version I tried, OLS stayed between 0.0719 and 0.0725. It barely reacts to how I build the parent education controls, unlike IV. I think that's because OLS uses all the variation in schooling, while IV only uses the small part driven by the nudge, so IV is touchier | Not really. Rounding and the control details are probably enough to explain this one |

**Ambiguities in the paper, to take into Section 3:**

- **Parent education classes:** the paper says "8 classes" of mother's and father's education but never says where the cutoffs are, so I had to pick my own (under 12 years, exactly 12, more than 12). I tried a few other cutoffs to see how much it mattered, and IV moved around between 0.1306 and 0.1335, which is enough to explain my gap with the paper's 0.132.
- **Missing parent education:** the paper says it fills missing values with the overall mean and adds dummies for imputed parents, and I did the same. What it doesn't say is how the imputed men are treated when the 8 interaction classes are built. In my version the filled-in values (about 10.3 years for mothers, 9.4 for fathers) land in the lowest class (under 12), so they're mixed in with parents who really have less than high school. The authors may have handled that differently.

- **Standard errors:** the paper doesn't say which kind it uses. I went with the default (unadjusted) and got 0.0469 for IV against the paper's 0.049. I haven't tried the robust version yet, so I don't know if it would get closer.
- **First-stage strength:** the paper doesn't report a first-stage F at all. With only nearc4 as the instrument mine is 16.19, which is above 10. For the headline version it's 8.41, but that's a joint test over three instruments, so I don't think it's the right one to judge weakness with.